# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [107]:
import os
import pandas as pd

# Get the path from environment variable
price_data_path = os.environ.get("PRICE_DATA")

print("PRICE_DATA points to:", price_data_path)


PRICE_DATA points to: ../../05_src/data/prices/


In [93]:
import pandas as pd
import os
import sys
from glob import glob

sys.path.append(os.getenv('SRC_DIR'))

from utils.logger import get_logger
_logs = get_logger(__name__)

In [94]:
# Step 1: import dotenv and os
import os
from dotenv import load_dotenv

# Step 2: Load environment variables from .env file
load_dotenv()

# Step 3: Access the PRICE_DATA variable
price_data_path = os.getenv("PRICE_DATA")

print("PRICE_DATA path:", price_data_path)


PRICE_DATA path: ../../05_src/data/prices/


In [104]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [96]:
import os

# Load the path from environment variable
price_data_path = os.environ.get("PRICE_DATA")

print("PRICE_DATA path:", price_data_path)


PRICE_DATA path: ../../05_src/data/prices/


In [105]:
import os
import glob

# Load the path from environment variable
price_data_path = os.environ.get("PRICE_DATA")

# Use glob to list all .parquet files in the directory
parquet_files = glob.glob(os.path.join(price_data_path, "*.parquet"))

print("Found parquet files:")
for f in parquet_files:
    print(f)


Found parquet files:


For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
import dask.dataframe as dd
from glob import glob
import os

# Load PRICE_DATA path
price_data_path = os.environ.get("PRICE_DATA")
parquet_files = glob(os.path.join(price_data_path, "*.parquet*"))

# Load Parquet files into a Dask DataFrame
df = dd.read_parquet(parquet_files)

# --- Add lags per ticker ---
# 1-day lag for Close
df["Close_lag_1"] = df.groupby("ticker")["Close"].shift(1)

# 1-day lag for Adj_Close
df["Adj_Close_lag_1"] = df.groupby("ticker")["Adj_Close"].shift(1)

# Preview results
print(df[["ticker", "Close", "Close_lag_1", "Adj_Close", "Adj_Close_lag_1"]].head())


In [ ]:
# Compute daily returns
df["returns"] = (df["Close"] / df["Close_lag_1"]) - 1

# Preview the result
print(df[["ticker", "Close", "Close_lag_1", "returns"]].head())


In [ ]:
# Compute daily high-low range
df["hi_lo_range"] = df["High"] - df["Low"]

# Preview the result
print(df[["ticker", "High", "Low", "hi_lo_range"]].head())


In [ ]:
# Assign the final DataFrame to dd_feat
dd_feat = df

# Preview the first few rows
print(dd_feat.head())


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Convert Dask DataFrame to pandas DataFrame
df_pd = dd_feat.compute()

# Preview the first few rows
print(df_pd.head())


In [ ]:
# Ensure data is sorted by ticker and date
df_pd = df_pd.sort_values(by=["ticker", "date"])

# Compute 10-day moving average of returns per ticker
df_pd["returns_ma_10"] = (
    df_pd.groupby("ticker")["returns"]
    .rolling(10)
    .mean()
    .reset_index(level=0, drop=True)
)

# Preview the result
print(df_pd[["ticker", "date", "returns", "returns_ma_10"]].head(15))


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

Was it necessary to convert to pandas?
Not really. Pandas simplifies rolling calculations, but Dask can also compute moving averages directly using .groupby().rolling().mean(). Converting to pandas is mainly for convenience or easier debugging.

Would it have been better to do it in Dask? Why?
Yes, especially for large datasets. Dask supports out-of-core computation and parallel processing, allowing rolling averages to be calculated without loading the entire dataset into memory. Converting to pandas could cause memory issues for large datasets.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.